# 00 Planung und Probebilder

**Wann:** vor der Aufnahmereihe. **Ergebnis:** Plan (Belichtung, Fokus/Defokus, Aperturen, Meldefilter) und Sitzung, die NB01–NB03 automatisch lesen.

### Welcher Fall sind Sie? — das legt Schritt 1 fest

| Fall | Kamera / Aufbau | Meldefilter | Besonderheit |
|:---|:---|:---|:---|
| **A** Mono + Standardfilter | Mono-CCD/CMOS, Filter V/B/R | `V`, `B`, `R` | Transformation (NB08) wünschenswert, `TRANS=NO` zulässig |
| **B** Mono + RGB-Filterrad | Mono-Kamera, RGB-Filter | `TB`, `TG`, `TR` | je Filter eigene Probebilder; Transformation (NB08) vorgesehen |
| **C** Farbkamera (OSC) | Gekühlte Farbkamera, Bayer | `TG`, `TB`, `TR` | FWHM im Rohbild ≥ 8–10 px; Kalibrierung ohne Debayern; NB08 |
| **D** DSLR / spiegellos | RAW (CR2, ARW …) | `TG`, `TB`, `TR` | wie C; zusätzlich RAW-Kompression/Profil prüfen (NB07) |

Für alle: Sterne unter **75 % des Maximalwerts**, Aperturradius **1,2–1,5 × FWHM**. Nur Bayer: FWHM ≥ 8–10 px.

### Schritte

| Schritt | Was | Ergebnis |
|:---|:---|:---|
| **0** | Vorbedingungen (Python, Pakete, ASTAP, Siril); **0b einmalig** Beobachter/Standort/Kamera | Tabelle ✓/✗; `observer.json` |
| **1** | Fall wählen (A–D), Ziel, Kamera, Sitzungsordner | Meldefilter, `OBSTYPE`, Sitzung |
| **2** | Standort, Sichtbarkeit, Mond; Bereitschaft (NB07, NB08?) | Zeitfenster, Status |
| **2c** | **Probebilder aufnehmen** (am Teleskop) und ablegen — Anleitung je Fall | Ordner `probe/`, `probe/scharf/` |
| **3–4** | Probebilder auswerten: Sättigung, FWHM, Apertur, Defokus, Warnungen | Plan |
| **5** | Plan speichern | `PHOTOMETRIE_PY_DATA/plans/*.json` |

**Reihenfolge:** einmalig **NB07** (Kamera) → **NB00** (hier) → Aufnahme → **NB01** (Kalibrierung/Comps) → **NB02** (Messung) → **NB03** (Bericht); Fälle B–D zusätzlich **NB08** (Transformation).  
**Kein Teleskop?** In Schritt 1 Modus „Demo" oder „LCO-Testdaten" wählen — Schritt 2c entfällt.

## Schritt 0 — Vorbedingungen

Prüft Python (≥ 3.12), die Pakete, **ASTAP** (Plate-Solving echter Bilder), Siril (optional), den Datenordner, `observer.json` (Standort) und die **KI-Wege** (optional). Dieselbe Prüfung wie in NB01 (`varphot.readiness`). ✗ = Hindernis, · = optional/fehlt.

In [ ]:
from IPython.display import HTML, display
from varphot import software_checks
from varphot.readiness import render_html
PRE = software_checks()
display(HTML(render_html(PRE, "Vorbedingungen — Notebook 00 Planung")))
ASTAP_OK = next((c.ok for c in PRE if c.label == "Programm ASTAP"), False)

## Schritt 0b — Formular „Beobachter, Standort, Kamera“ (einmalig)

**1. Zelle unten ausführen (▶) — das Formular erscheint darunter. 2. Ausfüllen wie in der Tabelle. 3. „Speichern“ klicken, bis „✓ gespeichert: …“ erscheint.**

| Feld | **Testlauf** (LCO-Testdaten / Demo) | **Eigene Aufnahmen** (einmalig, danach überspringen) |
|---|---|---|
| **Standort** | **nichts ändern**, nicht speichern — „LCO McDonald (Testdaten delta Cep)“ ist schon gewählt | Ihren Standort wählen, oder „➕ Neuer Standort …“ → Name, Breite (° N), Länge (° O; West = −), Höhe (m) |
| **AAVSO-Code** | egal (Testdaten gehen immer als `TST01` hinaus) | Ihr Beobachtercode, z. B. `ABCD` |
| **Kamera wählen** | egal — Testdaten haben ihre eigene Kamera | Profil/Vorlage wählen (hinter dem Namen: **Mono** oder **Farbe (Bayer …)**) oder „➕ Neue Kamera …“ |
| **Gain · Ausleserauschen · Dunkelstrom** | nicht nötig | nur wenn bekannt (leer = unbekannt; **NB07** misst sie) |

Die Daten liegen in `PHOTOMETRIE_PY_DATA/observer.json` (nicht im Repo). **Alles schon eingetragen? → direkt weiter zu Schritt 1.**

In [ ]:
import json
import ipywidgets as widgets
from pathlib import Path
from IPython.display import display
from dataclasses import replace
from varphot.hints import bind, nb00_observer_hints
from varphot import PRESETS
from varphot.observer import (Camera, Observer, Site, all_sites, camera_kind, camera_label, config_path,
                              load_config, save_config, upsert_camera, upsert_site)

_obs, _own_sites = load_config()
if _obs.camera.model and all((c.model, c.gain_setting) != (_obs.camera.model, _obs.camera.gain_setting) for c in _obs.cameras):
    _obs.cameras.append(_obs.camera)           # aeltere Datei: die einzelne aktive Kamera wird zum ersten Profil
_sites = all_sites(_own_sites)
_by_name = {s.name: s for s in _sites}
_NEW = "➕ Neuer Standort …"

_st = {"description_width": "150px"}
_lay = widgets.Layout(width="520px")
_num = lambda desc, val, **kw: widgets.FloatText(description=desc, value=val if val is not None else 0.0,
                                                 style=_st, layout=_lay, **kw)

w_site   = widgets.Dropdown(description="Standort:", options=[s.name for s in _sites] + [_NEW],
                            value=(_obs.site_name if _obs.site_name in _by_name else
                                   (_sites[0].name if _sites else _NEW)), style=_st, layout=_lay)
w_sname  = widgets.Text(description="Name:", style=_st, layout=_lay)
w_lat    = widgets.BoundedFloatText(description="Breite (° N):", min=-90, max=90, step=0.0001, style=_st, layout=_lay)
w_lon    = widgets.BoundedFloatText(description="Länge (° O, W = −):", min=-180, max=180, step=0.0001, style=_st, layout=_lay)
w_hgt    = widgets.FloatText(description="Höhe (m):", style=_st, layout=_lay)
w_code   = widgets.Text(description="AAVSO-Code:", value=_obs.aavso_code, placeholder="z. B. ABCD", style=_st, layout=_lay)
_CAM_NEW = "➕ Neue Kamera …"
_cam_opts = ([(f"Profil: {camera_label(c)} · {camera_kind(c)}", ("profil", i)) for i, c in enumerate(_obs.cameras)] +
             [(f"Vorlage: {pr.model} · {camera_kind(pr)}", ("vorlage", k)) for k, pr in PRESETS.items() if k != "eigene"] + [(_CAM_NEW, ("neu", ""))])
_active = next((("profil", i) for i, c in enumerate(_obs.cameras) if (c.model, c.gain_setting) == (_obs.camera.model, _obs.camera.gain_setting)
                and _obs.camera.model), ("neu", ""))
w_camsel = widgets.Dropdown(description="Kamera wählen:", options=_cam_opts, value=_active, style=_st, layout=_lay)
w_cmodel = widgets.Text(description="Kamera-Modell:", value=_obs.camera.model, style=_st, layout=_lay)
w_ctype  = widgets.Dropdown(description="Kamera-Typ:", options=["CMOS", "CCD", "DSLR", "Sonstige"],
                            value=_obs.camera.cam_type if _obs.camera.cam_type in ("CMOS", "CCD", "DSLR", "Sonstige") else "CMOS",
                            style=_st, layout=_lay)
w_gain   = widgets.Text(description="Gain (e⁻/ADU):", value="" if _obs.camera.gain_e_per_adu is None else str(_obs.camera.gain_e_per_adu), placeholder="leer = unbekannt", style=_st, layout=_lay)
w_rn     = widgets.Text(description="Ausleserauschen (e⁻):", value="" if _obs.camera.readout_noise_e is None else str(_obs.camera.readout_noise_e), placeholder="leer = unbekannt", style=_st, layout=_lay)
w_dark   = widgets.Text(description="Dunkelstrom (e⁻/px/s):", value="" if _obs.camera.dark_current_e_per_s is None else str(_obs.camera.dark_current_e_per_s), placeholder="leer = unbekannt", style=_st, layout=_lay)
w_dtemp  = widgets.Text(description="… bei Temperatur (°C):", value="" if _obs.camera.dark_temp_c is None else str(_obs.camera.dark_temp_c), style=_st, layout=_lay)
w_lin    = widgets.Text(description="Linearität:", value=_obs.camera.linearity, placeholder="z. B. 99 % bis 40000 ADU", style=_st, layout=_lay)
w_btn    = widgets.Button(description="Speichern", button_style="success", icon="save")
w_msg    = widgets.HTML()

def _fill_site(*_):
    s = _by_name.get(w_site.value)
    new = s is None
    w_sname.value = "" if new else s.name
    w_lat.value, w_lon.value, w_hgt.value = (0.0, 0.0, 0.0) if new else (s.lat_deg, s.lon_deg, s.height_m)
    for w in (w_sname, w_lat, w_lon, w_hgt):
        w.disabled = not new and s in [b for b in _sites if b not in _own_sites]   # Testdaten-Standorte sind fest
w_site.observe(_fill_site, names="value")
_fill_site()

def _fill_cam(*_):
    kind, ref = w_camsel.value
    if kind == "profil":
        c = _obs.cameras[ref]
        vals = (c.model, c.cam_type, c.gain_e_per_adu, c.readout_noise_e, c.dark_current_e_per_s, c.dark_temp_c, c.linearity)
    elif kind == "vorlage":                      # Vorlage liefert nur Modell/Typ; Messwerte kommen aus NB07
        pr = PRESETS[ref]
        vals = (pr.model, pr.cam_type, None, None, None, None, "")
    else:
        vals = ("", "CMOS", None, None, None, None, "")
    m, t, g, r, d, dt, lin = vals
    w_cmodel.value = m
    w_ctype.value = "DSLR" if t.startswith("DSLR") else (t if t in w_ctype.options else "CMOS")
    for w, v in ((w_gain, g), (w_rn, r), (w_dark, d), (w_dtemp, dt)):
        w.value = "" if v is None else str(v)
    w_lin.value = lin
w_camsel.observe(_fill_cam, names="value")
_fill_cam()

def _base_camera():
    """Ausgangsprofil fuer das Speichern: gewaehltes Profil bzw. Vorlage; bewahrt die NB07-Felder (Bayer-Muster, Bittiefe, Maximalwert …)."""
    kind, ref = w_camsel.value
    if kind == "profil":
        return _obs.cameras[ref]
    if kind == "vorlage":
        pr = PRESETS[ref]
        return Camera(bayer_pattern=pr.bayer_pattern, bit_depth=pr.bit_depth)
    return Camera()

def _opt(txt):
    txt = txt.strip().replace(",", ".")
    return float(txt) if txt else None

def _save(_):
    global _own_sites, _sites, _by_name
    try:
        site = Site(w_sname.value, w_lat.value, w_lon.value, w_hgt.value)
        cam = replace(_base_camera(), model=w_cmodel.value.strip(), cam_type=w_ctype.value, gain_e_per_adu=_opt(w_gain.value),
                      readout_noise_e=_opt(w_rn.value), dark_current_e_per_s=_opt(w_dark.value), dark_temp_c=_opt(w_dtemp.value),
                      linearity=w_lin.value.strip())
        obs = Observer(w_code.value, site.name, cam, upsert_camera(_obs.cameras, cam) if cam.model else list(_obs.cameras))
        if site.name not in {s.name for s in _sites} or site.name in {s.name for s in _own_sites}:
            _own_sites = upsert_site(_own_sites, site)
        path = save_config(obs, _own_sites)
    except ValueError as e:
        w_msg.value = f'<span style="color:#f44336">✗ {e}</span>'
        return
    _sites = all_sites(_own_sites); _by_name = {s.name: s for s in _sites}
    w_site.options = [s.name for s in _sites] + [_NEW]; w_site.value = site.name
    _o2, _ = load_config(); _obs.cameras[:] = _o2.cameras           # Profilliste aktuell halten
    w_msg.value = f'<span style="color:#4caf50">✓ gespeichert: {path}</span>'
w_btn.on_click(_save)

w_hint = widgets.HTML()
def _site_hint():
    s = _by_name.get(w_site.value)
    is_test = s is not None and s not in _own_sites
    kind, ref = w_camsel.value
    cam = _obs.cameras[ref] if kind == "profil" else (PRESETS[ref] if kind == "vorlage" else None)
    return nb00_observer_hints(w_site.value, is_test, s is None, w_code.value, camera_label(cam) if kind == "profil" else (cam.model if cam else ""),
                               camera_kind(cam) if cam else "")
bind(w_hint, [w_site, w_code, w_camsel], _site_hint)
display(widgets.VBox([
    w_hint,
    widgets.HTML("<b>Standort</b>"), w_site, w_sname, w_lat, w_lon, w_hgt,
    widgets.HTML("<b>AAVSO</b>"), w_code,
    widgets.HTML("<b>Kamera (Detector Details)</b>"), w_camsel, w_cmodel, w_ctype, w_gain, w_rn, w_dark, w_dtemp, w_lin,
    w_btn, w_msg,
]))


## Schritt 1 — Formular „Fall, Ziel, Kamera“

**1. Zelle unten ausführen (▶) — das Formular erscheint darunter. 2. Ausfüllen wie in der Tabelle. 3. Nächste Zelle ausführen.**

> **Testlauf mit den LCO-Testdaten?** Nur **Modus = „LCO-Testdaten“** wählen — Ziel, Methode, Kamera und Meldefilter werden dann **automatisch vorgegeben** (grau). Alles andere so lassen.

| Feld | **Testlauf** (Modus „LCO-Testdaten“) | **Eigene Aufnahmen** (Modus „Echte Probebilder“) |
|---|---|---|
| **Modus** | **LCO-Testdaten** wählen | **Echte Probebilder** (ohne Daten zum Ansehen: *Demo*) |
| **Sitzungsordner** | so lassen — vorbelegt mit `PHOTOMETRIE_PY_DATA/sessions/<heutiges Datum>`, wird neu angelegt | Siril-Arbeitsordner dieser Nacht (mit `lights/`, `darks/`, `flats/`, `biases/`, `probe/`) |
| **Ziel** | vorgegeben: `del Cep` (δ Cephei — die LCO-Testdaten zeigen diesen Stern) | Ihr Stern; Name wie in SIMBAD/VSX |
| **V (hellste Phase)** | leer | leer; nur eintragen, wenn SIMBAD keinen Wert liefert |
| **Methode** | vorgegeben: *Monochrom + Standardfilter* (Fall A) | Ihr Fall A–D (Tabelle oben) |
| **Kamera** | vorgegeben: ein **Mono**-Profil (welches, ist egal) | Ihre Kamera — hinter dem Namen steht **Mono** oder **Farbe (Bayer …)**; passend zur Methode wählen |
| **Meldefilter** | vorgegeben: `V` | Fall A: `V`, `B` oder `R`; Fälle B–D: `TG`, `TB`, `TR` |
| **Brennweite [mm]** | leer | Ihre Brennweite (für Feldgröße und Plate-Solve) |

*„Mono“/„Farbe“ wird aus dem Kameraprofil abgeleitet (Bayer-Muster); das hat mit der AAVSO nichts zu tun, online ist nichts anzupassen.*

In [ ]:
import os, datetime
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from varphot import (METHODS, ObservingPlan, PRESETS, aperture_from_fwhm, fwhm_summary, green_plane, night_altitudes, recommend_exposure,
                     save_plan, star_metrics, get_preset, field_of_view_deg)
from varphot.plan import check_field_stars, comp_v_min, plane_wcs
from varphot.observer import all_sites, camera_kind, camera_label, data_dir, load_config
from varphot.hints import bind, nb00_hints
from varphot.plan import SAFE_FRACTION, MIN_RAW_FWHM_BAYER
from varphot import defocus_tradeoff, defocus_needed, defocus_image, camera_readiness, std_field_visibility
from varphot.readiness import render_html
from IPython.display import HTML
from varphot.testdata import testdata_dir
from varphot.vsp import resolve_target
from varphot.batch import read_frame

_obs, _own = load_config()
_cams = list(_obs.cameras)
if _obs.camera.model and all((c.model, c.gain_setting) != (_obs.camera.model, _obs.camera.gain_setting) for c in _cams):
    _cams.append(_obs.camera)
_cam_opts = [(f"Profil: {camera_label(c)} · {camera_kind(c)}", ("profil", i)) for i, c in enumerate(_cams)] + [(f"Vorlage: {p.model} · {camera_kind(p)}", ("vorlage", k)) for k, p in PRESETS.items() if k != "eigene"]
_st = {"description_width": "110px"}; _lay = widgets.Layout(width="560px")
w_target = widgets.Text(description="Ziel:", value=os.environ.get("VARPHOT_TARGET", "del Cep"), style=_st, layout=_lay)
w_vmax = widgets.Text(description="V (hellste Phase):", value=os.environ.get("VARPHOT_VMAX", ""), placeholder="leer = SIMBAD-Wert", style=_st, layout=_lay)
w_method = widgets.Dropdown(description="Methode:", options=[(v, k) for k, v in METHODS.items()], value=os.environ.get("VARPHOT_METHOD", "osc"), style=_st, layout=_lay)
w_cam = widgets.Dropdown(description="Kamera:", options=_cam_opts, style=_st, layout=_lay)
w_filter = widgets.Dropdown(description="Meldefilter:", options=["V", "B", "R", "TG", "TB", "TR"], value="TG", style=_st, layout=_lay)
def _meth(*_):
    w_filter.options = ["V", "B", "R"] if w_method.value == "mono" else ["TG", "TB", "TR", "V", "B", "R"]
    w_filter.value = "V" if w_method.value == "mono" else "TG"
w_method.observe(_meth, names="value"); _meth()
w_mode = widgets.Dropdown(description="Modus:", options=[("Demo (synthetisch, nichts wird gespeichert)", "demo"), ("Echte Probebilder (Ordner probe/)", "real"), ("LCO-Testdaten (echte Bilder zum Ausprobieren)", "lco")], value=os.environ.get("VARPHOT_NB00_MODE", "demo"), style=_st, layout=_lay)
_ws0 = os.environ.get("VARPHOT_WORKSPACE") or str((data_dir() / "sessions" if data_dir().is_dir() else Path.home() / "Astro") / f"{datetime.date.today():%Y-%m-%d}")
w_ws = widgets.Text(description="Sitzungsordner:", value=_ws0, placeholder="= Siril-Arbeitsordner (lights/, darks/, flats/, biases/, probe/)", style=_st, layout=_lay)
w_focal = widgets.Text(description="Brennweite [mm]:", value=os.environ.get("VARPHOT_FOCAL_MM", ""), placeholder="z. B. 1645 (C9.25 + Reducer)", style=_st, layout=_lay)
w_info = widgets.HTML()
def _cam_obj(v):
    kind, ref = v
    return _cams[ref] if kind == "profil" else PRESETS[ref]
def _is_mono(v):
    return camera_kind(_cam_obj(v)) == "Mono"
def _mode(*_):
    lco = w_mode.value == "lco"
    if lco:                                   # Testdaten: alles vorgegeben (grau) — nichts zu entscheiden
        w_target.value, w_vmax.value, w_focal.value = "del Cep", "", ""
        w_method.value = "mono"; w_filter.value = "V"
        mono = next((v for _, v in _cam_opts if _is_mono(v)), None)
        if mono:
            w_cam.value = mono
    for w in (w_target, w_vmax, w_method, w_cam, w_filter, w_focal):
        w.disabled = lco
w_mode.observe(_mode, names="value"); _mode()
bind(w_info, [w_mode, w_ws, w_method, w_cam, w_filter],
     lambda: nb00_hints(w_mode.value, w_method.value, w_filter.value, camera_kind(_cam_obj(w_cam.value)), w_ws.value))
display(widgets.VBox([w_mode, w_info, w_ws, w_target, w_vmax, w_method, w_cam, w_filter, w_focal]))

---
## ▶ Ab hier automatisch

Ab dieser Zelle ist **nichts mehr einzugeben**: Pfeil neben der Zelle → **„Diese Zelle und alle folgenden“** ausführen. Das Notebook wertet die Probebilder aus und endet mit dem gespeicherten Plan (Schritt 5).

### Auswahl übernehmen

Löst das Ziel über SIMBAD auf (bei fehlendem Netz: `RA_DEG`/`DEC_DEG`/`V_MEAN` unten von Hand setzen) und liest das Kamera-Profil.

In [ ]:
from varphot import Session
from varphot.session import session_tag
from varphot.workflow import prepare_lco_demo
MODE_SEL = w_mode.value
MODE = "demo" if MODE_SEL == "demo" else "real"
TARGET, METHOD, AAVSO_FILTER = w_target.value.strip(), w_method.value, w_filter.value
if MODE_SEL == "lco":
    TARGET, METHOD, AAVSO_FILTER = "del Cep", "mono", "V"
SESSION = Session.open(w_ws.value.strip(), target=TARGET, method=METHOD, aavso_filter=AAVSO_FILTER, source={"demo": "demo", "real": "own", "lco": "lco-demo"}[MODE_SEL])
if SESSION.notes.get("zurueckgesetzt"):
    print("⚠", SESSION.notes["zurueckgesetzt"])
if MODE != "demo":
    _made = SESSION.ensure_dirs()
    if MODE_SEL == "lco":
        _lco = prepare_lco_demo(SESSION)
        print(f"LCO-Demo: {len(_lco['probe'])} Probebilder + 1 scharfes Referenzbild verknüpft, Reihe in {_lco['frames_dir']}")
    print(f"Sitzungsordner {SESSION.workspace}" + (f" (neu angelegt: {', '.join(_made)})" if _made else ""))
kind, ref = w_cam.value
if kind == "profil":
    CAM, PRESET = _cams[ref], None
else:
    PRESET = get_preset(ref); CAM = None
MODEL = CAM.model if CAM else PRESET.model
_pre = PRESET or next((p for p in PRESETS.values() if p.model == MODEL), None)                    # Sensorformat und Pixelgröße aus der Vorlage (auch für gespeicherte Profile gleichen Modells)
FOCAL_MM = float(w_focal.value.replace(",", ".")) if w_focal.value.strip() else None
FIELD_DEG = list(field_of_view_deg(_pre.format_px, _pre.pixel_um, FOCAL_MM)) if (FOCAL_MM and _pre and _pre.pixel_um and _pre.format_px[0]) else []
if FIELD_DEG:
    print(f"Bildfeld aus dem Aufbau: {FIELD_DEG[0]:.2f}° × {FIELD_DEG[1]:.2f}° (Brennweite {FOCAL_MM:g} mm, {_pre.model}) — NB03 übernimmt es über den Plan")
else:
    print("Keine Feldgröße im Plan (Brennweite oder Sensorformat fehlt) — NB03 nutzt dann FIELD_DEG von Hand oder ein Referenzbild")
SETTING = CAM.gain_setting if CAM else next(iter(PRESET.settings))
SESSION.camera_model, SESSION.camera_setting, SESSION.obstype = MODEL, SETTING, ("CCD" if METHOD == "mono" else "DSLR")
PATTERN = (CAM.bayer_pattern if CAM else PRESET.bayer_pattern) or "RGGB"
OBSTYPE = "CCD" if METHOD == "mono" else "DSLR"                       # vorläufig: auch Mono + RGB-Filter zählt in der AAVSO-Forum-Zusammenfassung als "DSLR" (KONZEPT §11.2/§11.12)
try:
    T = resolve_target(TARGET)
    RA_DEG, DEC_DEG, V_MEAN = T["ra"], T["dec"], T.get("V")
except Exception as e:                                                  # offline: von Hand setzen
    print("SIMBAD nicht erreichbar:", e)
    RA_DEG, DEC_DEG, V_MEAN = 337.2928, 58.4152, 3.75                    # <- bei Bedarf eintragen (hier δ Cep)
V_MAX = float(w_vmax.value.replace(",", ".")) if w_vmax.value.strip() else V_MEAN
BITS = (CAM.bit_depth if CAM and CAM.bit_depth else None) or (PRESET.bit_depth if PRESET else None) or 14
OFFSET = CAM.offset_adu if CAM and CAM.offset_adu is not None else None
MAX_ADU = CAM.saturation_adu if CAM and CAM.saturation_adu else (2 ** BITS - 1)
if CAM and CAM.linear_max_adu:
    LIMIT, LIMIT_SRC = float(CAM.linear_max_adu), "gemessene Linearitätsgrenze (NB07)"
else:
    LIMIT, LIMIT_SRC = SAFE_FRACTION * MAX_ADU, f"{int(SAFE_FRACTION * 100)} % des Maximalwerts (Handbuch §4.5.2; kein NB07-Profil)"
print(f"Ziel {TARGET}: RA {RA_DEG:.4f}°, Dec {DEC_DEG:+.4f}°, V = {V_MEAN} (hellste Phase für die Planung: {V_MAX})")
print(f"Methode: {METHODS[METHOD]} → Meldefilter {AAVSO_FILTER}, OBSTYPE {OBSTYPE}")
print(f"Kamera: {MODEL} — {SETTING} ({'Profil' if CAM else 'Vorlage ohne Messung'}), Bayer {PATTERN if METHOD in ('osc', 'dslr') else '—'}, Maximalwert {MAX_ADU:.0f} ADU, "
      f"Obergrenze für Sterne {LIMIT:.0f} ADU [{LIMIT_SRC}]")
if CAM is None:
    print("HINWEIS: für diese Kamera gibt es noch kein NB07-Profil — Offset, Maximalwert und Linearität sind nominell.")

## Schritt 2 — Standort und Sichtbarkeit

Standort aus NB01. Grün = dunkel (Sonne < −12°) und Ziel über 30°. Der **Mond** wird mitgerechnet (Höhe, Abstand zum Ziel, beleuchteter Anteil): er hellt den Himmel auf (Himmelsrauschen, Gradienten im Himmelsring) — bei hellen Zielen wie δ Cep meist unkritisch, bei schwachen schon. "Mond ok" heißt: Mond unter dem Horizont oder Abstand > 40° bei Beleuchtung < 50 % (Startwerte). Ohne Wetter.

In [ ]:
SITE = next((s for s in all_sites(_own) if s.name == _obs.site_name), None)
MOON = {}
if SITE is None:
    print("Kein Standort gespeichert — NB01 Abschnitt 0 ausfüllen. Sichtbarkeit übersprungen.")
else:
    NA = night_altitudes(RA_DEG, DEC_DEG, SITE, step_min=30)
    ok = NA[NA["ok"]]
    print(f"Standort {SITE.name}: " + (f"Beobachtungsfenster (UTC) {ok['utc'].iloc[0]} bis {ok['utc'].iloc[-1]}, Luftmasse {ok['airmass'].min():.2f}–{ok['airmass'].max():.2f}" if len(ok) else "heute Nacht kein Fenster (dunkel und > 30°)"))
    MOON = {}
    if len(ok):
        okm = NA[NA["ok_moon"]]
        MOON = dict(illum=round(float(ok["moon_illum"].mean()), 2), sep_deg=round(float(ok["moon_sep"].mean())), up_fraction=round(float((ok["alt_moon"] > 0).mean()), 2),
                    window_without_moon_disturbance=int(len(okm)) * 30)
        print(f"Mond im Fenster: Beleuchtung {100 * MOON['illum']:.0f} %, mittlerer Abstand zum Ziel {MOON['sep_deg']}°, {100 * MOON['up_fraction']:.0f} % der Zeit über dem Horizont; "
              f"ungestörtes Fenster {MOON['window_without_moon_disturbance']} min")
        if len(okm) < len(ok):
            print("⚠ Mond stört in Teilen des Fensters" + (" (heller Mond nahe am Ziel)" if MOON["illum"] > 0.5 and MOON["sep_deg"] < 40 else ""))
    fig, ax = plt.subplots(figsize=(8, 2.8))
    ax.plot(range(len(NA)), NA["alt_target"], label="Ziel"); ax.plot(range(len(NA)), NA["alt_sun"], color="orange", label="Sonne"); ax.plot(range(len(NA)), NA["alt_moon"], color="gray", ls="--", label="Mond")
    ax.fill_between(range(len(NA)), -90, 90, where=NA["ok_moon"], color="g", alpha=0.18, label="Fenster (Mond ok)"); ax.fill_between(range(len(NA)), -90, 90, where=NA["ok"] & ~NA["ok_moon"], color="y", alpha=0.18, label="Fenster (Mond stört)"); ax.axhline(30, color="k", ls=":", lw=0.6)
    ax.set_xticks(range(0, len(NA), 4)); ax.set_xticklabels([u[11:] for u in NA["utc"][::4]], rotation=45); ax.set_ylim(-60, 90); ax.set_ylabel("Höhe [°]"); ax.legend(loc="upper right")
    plt.tight_layout(); plt.show()

### Schritt 2b — Bereitschaft: Kameraprofil (NB07), Transformation (NB08), Standardfelder

* **NB07** liefert Gain, Ausleserauschen, Linearitätsgrenze und die RAW-Kompressionsprüfung der Kamera. Ohne Profil gilt die 75-%-Regel mit dem nominellen Maximalwert.
* **NB08** liefert die Transformationskoeffizienten (nur bei Farbkamera/RGB-Filter nötig). Ohne sie werden die Aufnahmen als `TG`/`TB`/`TR` **untransformiert** gemeldet.
* **Standardfelder** (M 67, M 11, NGC 7790 …) sind für NB08 nötig; die Tabelle zeigt, welche heute Nacht am Standort hoch genug stehen (Stunden mit dunklem Himmel und > 30°).

In [ ]:
READY = camera_readiness(CAM, METHOD, MODEL, SETTING)
display(HTML(render_html(READY, "Bereitschaft: Kameraprofil und Transformation")))
STD_FIELD = None
if METHOD in ("osc", "dslr", "monorgb"):
    if SITE is None:
        print("Kein Standort — Sichtbarkeit der Standardfelder übersprungen (NB01 Abschnitt 0).")
    else:
        SF = std_field_visibility(SITE, min_alt=30)
        display(SF)
        best = SF[SF["ok"]]
        STD_FIELD = None if best.empty else str(best.iloc[0]["field"])
        print(f"Standardfeld für NB08: {STD_FIELD} ({best.iloc[0]['hours']:.1f} h, bis {best.iloc[0]['alt_max']:.0f}° hoch)" if STD_FIELD else "⚠ Heute Nacht kein Standardfeld sichtbar — anderes Datum prüfen oder Feld per eigener CSV (NB08).")
READINESS = dict(nb07=bool(READY[0].ok), nb08=(bool(READY[1].ok) if len(READY) > 1 else None), std_field=STD_FIELD)

## Schritt 2c — Probebilder aufnehmen und ablegen (am Teleskop, vor der Reihe)

> **Ziel:** in 10 Minuten wissen, welche Belichtung geht und wie unscharf gestellt werden muss. Gemacht wird das **mit dem Aufbau der späteren Reihe** — gleiche Kamera, gleicher Filter, gleiche Brennweite, gleicher Gain.

**Was aufnehmen (alle Fälle):**

| # | Aufnahme | Wie | Zweck |
|---|---|---|---|
| **1–3** | **Messbilder** desselben Feldes | Ziel im Bild, in seiner **hellsten Phase** planen (Veränderliche werden heller!); Belichtung **0,5× · 1× · 2×** der geschätzten Zeit; Fokus/Defokus **wie geplant** | Sättigung, FWHM, längste zulässige Belichtung |
| **4** | **Scharfes** Bild | gleiche Belichtung wie Messbild 2, **scharf fokussiert** | Plate-Solve-Test und „scharf gegen defokussiert“ (Schritt 3c) |

**Je nach Fall zusätzlich:**

| Fall | Zusätzlich beachten |
|---|---|
| **A  Mono + Filter** | Probebilder **im Meldefilter** (z. B. V). |
| **B  Mono + RGB-Filterrad** | Probebilder im **Grünfilter (TG)**; die anderen Filter später mit gleicher Belichtung *prüfen*, nicht neu planen. |
| **C  Farbkamera (OSC)** | **RAW/FITS ohne Debayern** speichern (CFA). Sterne dürfen **nicht zu scharf** sein: FWHM im Rohbild ≥ 8–10 px → ggf. defokussieren. |
| **D  DSLR** | **RAW** (CR2 …); zuerst in Siril ohne Debayern nach FITS wandeln (NB07 Schritt 1, oder Befehl `convertraw`). |

**Wohin damit** (im Sitzungsordner aus Schritt 1):

```
<Sitzungsordner>/
├── probe/            ← Messbilder 1–3
│   └── scharf/       ← Bild 4 (scharf fokussiert)
```

**Checkliste vor Schritt 3:**
- ☐ 3 Messbilder in `probe/`, gleiches Feld, Ziel sichtbar
- ☐ 1 scharfes Bild in `probe/scharf/`
- ☐ Fall B–D: RAW nicht debayert (CFA erhalten)
- ☐ Modus in Schritt 1 = **Echte Probebilder**

*Sie wollen es nur ausprobieren?* Modus „Demo“ oder „LCO-Testdaten“ — keine eigenen Bilder nötig.

## Schritt 3 — Probebilder auswerten

Je Bild: hellster Stern (Spitzenwert mit Offset, gesättigt?) und **FWHM** der mittelhellen, ungesättigten Sterne (Gauß-Anpassung). Bei Bayer-Kameras wird auf der **Grünebene** gemessen (beide Grün gemittelt, halbe Auflösung);
die Rohbild-FWHM ist doppelt so groß und muss ≥ 8–10 px sein. Im Modus `real` stehen die Probebilder als FITS (bei RAW: zuerst mit Siril ohne Debayern wandeln, siehe NB07) in `TEST_DIR`.

In [ ]:
TEST_DIR = Path(os.environ.get("VARPHOT_TEST_DIR") or SESSION.path("probe"))
BAYER = METHOD in ("osc", "dslr")                                  # nur Bayer-Kameras: Grünebene, FWHM im Rohbild ≥ 8–10 px

STARS_DEMO = [(400, 500, 3800.0), (150, 200, 900.0), (600, 300, 600.0), (250, 800, 450.0), (650, 850, 300.0), (120, 650, 220.0), (700, 120, 160.0)]   # (y, x, Spitzenwert ADU/s); erster = Ziel
def synth_frame(fwhm_raw, exptime, seed):
    """Syntetisches Probebild: Sternfeld, Bayer-Empfindlichkeiten (R 0,8 / G 1,0 / B 0,7), Ziel (der hellste Stern) wird mit der Belichtung heller."""
    rng = np.random.default_rng(seed)
    ny, nx = 800, 1000
    yy, xx = np.mgrid[0:ny, 0:nx]
    img = 100.0 + rng.normal(0, 4.0, (ny, nx))
    s = fwhm_raw / 2.355
    for y0, x0, rate in STARS_DEMO:                                              # rate = Spitzenwert in ADU je Sekunde
        img += rate * exptime * np.exp(-((xx - x0) ** 2 + (yy - y0) ** 2) / (2 * s * s))
    if BAYER:
        q = np.ones((ny, nx)); q[0::2, 0::2] = 0.8; q[1::2, 1::2] = 0.7
        img = 100.0 + (img - 100.0) * q
    return np.clip(np.rint(img), 0, MAX_ADU)

frames = []
if MODE == "demo":
    FW_DEMO = 6.0 if BAYER else 4.5                                         # Bayer-Demo absichtlich zu scharf → Warnung
    for k, t in enumerate((0.5, 1.0, 2.0, 4.0)):
        frames.append((f"synthetisch_{t:g}s", t, synth_frame(FW_DEMO, t, 10 + k)))
    print(f"DEMO: {len(frames)} synthetische Probebilder (Rohbild-FWHM {FW_DEMO} px)")
else:
    from varphot.camera import exposure_from_header
    from varphot import siril
    if not any(TEST_DIR.glob("*.fit*")) and siril.sequence_kind(TEST_DIR) == "raw":        # RAW-Probebilder: per Siril ohne Debayern nach FITS wandeln
        _script = siril.write_script(SESSION.path("scripts", "probe_convert.ssf"), siril.build_convert_script("raw", "probe", "fits"))
        _ok, _log = siril.run_script(_script, TEST_DIR)
        print("Siril-Wandlung der RAW-Probebilder:", "fertig" if _ok else f"nicht ausgeführt — {_log}")
    _files = sorted(list(TEST_DIR.glob("*.fit*")) + list((TEST_DIR / "fits").glob("*.fit*")))
    if not _files:
        raise FileNotFoundError(f"Keine Probebilder in {TEST_DIR}. 3–5 Aufnahmen des Feldes (FITS oder RAW) dorthin legen, ein scharfes Bild nach {TEST_DIR / 'scharf'}; oder Modus Demo/LCO wählen.")
    for p in _files:
        img, hdr, _ = read_frame(p)
        t = exposure_from_header(hdr)
        if t is None:
            raise RuntimeError(f"{p.name}: keine Belichtungszeit in der Kopfzeile (EXPTIME/EXPOSURE)")
        frames.append((p.name, t, img))
    print(f"{len(frames)} Probebilder aus {TEST_DIR}")

rows, FW_ALL, DFS = [], [], {}
for name, t, img in frames:
    work = green_plane(img, PATTERN) if BAYER else np.asarray(img, float)
    lim = MAX_ADU
    df = star_metrics(work, max_adu=lim, fwhm_guess=3.0 if BAYER else 4.0)
    fw = fwhm_summary(df)
    DFS[name] = (df, work, fw)
    top = df.iloc[0] if len(df) else None
    off = OFFSET if OFFSET is not None else float(np.median(work))
    rows.append(dict(Bild=name, Belichtung_s=t, hellster_Stern_ADU=None if top is None else round(top["raw_max"]), gesättigt=None if top is None else bool(top["saturated"]),
                     Anteil_am_Maximum=None if top is None else round(top["raw_max"] / MAX_ADU, 2), FWHM_px=round(fw["fwhm"], 2), n_Sterne=fw["n"], Offset=round(off)))
    if fw["n"]:
        FW_ALL.append(fw["fwhm"])
TAB = pd.DataFrame(rows)
display(TAB)

## Schritt 3b — Jeden Katalogstern prüfen (Ziel, Comps, Check)

Nicht nur der hellste Stern des Bildes zählt, sondern **jeder Stern, der in die Photometrie eingeht**: keiner darf über die Obergrenze (Handbuch §4.5.2), und jeder Comp braucht genug Signal (**S/N ≥ 100**, §6.3.1). Das Ziel wird für seine hellste Phase gerechnet.
Dazu braucht es (1) die Positionen der Sterne im Probebild (WCS aus der Kopfzeile oder per ASTAP, bei Bayer-Bildern auf die Grünebene umgerechnet) und (2) einen Katalog der Sterne (`field_config.json` aus NB03, `VARPHOT_FIELD_CONFIG`).
Ergebnis: je Stern die längste zulässige Belichtung, das S/N und die **hellste zulässige Comp-Helligkeit** `V_comp_min` (für NB03).

In [ ]:
FIELD, T_MAX_FIELD, V_COMP_MIN = None, None, None
TARGET_IS_BRIGHTEST = True            # das Ziel ist der hellste Stern im Feld (δ Cep: ja)
ref = None
FWHM_TMP = float(np.median(FW_ALL)) if FW_ALL else 4.0
okf = TAB[(TAB["gesättigt"] == False) & TAB["hellster_Stern_ADU"].notna()]
if len(okf):
    ref = okf.sort_values("hellster_Stern_ADU", ascending=False).iloc[0]            # ungesättigtes Bild mit dem besten Signal
if ref is None:
    print("kein ungesättigtes Probebild — Sternprüfung übersprungen")
else:
    name_ref, t_ref, img_ref = next(f for f in frames if f[0] == ref["Bild"])
    if MODE == "demo":
        from astropy.wcs import WCS
        w_raw = WCS(naxis=2); w_raw.wcs.crpix = [500, 400]; w_raw.wcs.crval = [RA_DEG, DEC_DEG]; w_raw.wcs.ctype = ["RA---TAN", "DEC--TAN"]
        w_raw.wcs.cdelt = [-1.44 / 3600, 1.44 / 3600]
        rows = []
        for k, (y0, x0, rate) in enumerate(STARS_DEMO):
            ra, dec = w_raw.pixel_to_world_values(x0, y0)
            rows.append(dict(id="Ziel" if k == 0 else f"C{k}", role="target" if k == 0 else "comp", ra=float(ra), dec=float(dec), V=V_MEAN - 2.5 * np.log10(rate / STARS_DEMO[0][2])))
        CATSTARS = pd.DataFrame(rows)
    else:
        import json
        from varphot.vsp import vsp_table  # noqa: F401
        fc_path = Path(os.environ.get("VARPHOT_FIELD_CONFIG", f"../data/fields/{TARGET.replace(' ', '_')}_field_config.json"))
        if not fc_path.exists():
            CATSTARS, w_raw = None, None
            print("Keine field_config.json gefunden:", fc_path, "— NB03 ausführen oder VARPHOT_FIELD_CONFIG setzen; Sternprüfung übersprungen.")
        else:
            rec = json.loads(fc_path.read_text())
            rows = [dict(id="Ziel", role="target", ra=rec["ziel"]["ra"], dec=rec["ziel"]["dec"], V=V_MEAN)]
            rows += [dict(id=str(s["label"]), role=s["rolle"], ra=s["ra"], dec=s["dec"], V=s["V"]) for s in rec["sterne"] if s["rolle"] in ("comp", "check")]
            CATSTARS = pd.DataFrame(rows)
            _img, _hdr, w_raw = read_frame(TEST_DIR / name_ref)
            if w_raw is None:
                from varphot.astrometry import solve_image
                sr = solve_image(TEST_DIR / name_ref)
                w_raw = sr.wcs
                print("WCS per ASTAP gelöst:", sr.source)
    if CATSTARS is not None and w_raw is not None:
        work = green_plane(img_ref, PATTERN) if BAYER else np.asarray(img_ref, float)
        w_use = plane_wcs(w_raw, (0.5, 0.5)) if BAYER else w_raw
        offv = OFFSET if OFFSET is not None else float(np.median(work))
        gain_v = (CAM.gain_e_per_adu if CAM and CAM.gain_e_per_adu else 1.0); rn_v = (CAM.readout_noise_e if CAM and CAM.readout_noise_e else 0.0)
        if not (CAM and CAM.gain_e_per_adu):
            print("HINWEIS: kein Gain/Rauschen aus NB07 — S/N-Angaben sind nur grob (Gain 1, kein Ausleserauschen).")
        T_REC_TMP = float(ref["Belichtung_s"])
        FIELD = check_field_stars(work, w_use, CATSTARS, exptime_s=float(ref["Belichtung_s"]), limit_adu=LIMIT, offset=offv, aperture=aperture_from_fwhm(FWHM_TMP),
                                  gain=gain_v, read_noise_e=rn_v, max_adu=MAX_ADU, snr_min=100.0)
        if TARGET_IS_BRIGHTEST:
            f_phase = 10 ** (0.4 * (V_MEAN - V_MAX)) if (V_MAX is not None and V_MEAN is not None and V_MAX < V_MEAN) else 1.0
            FIELD.loc[FIELD["role"] == "target", "t_max_s"] /= f_phase                 # Ziel in der hellsten Phase
        T_MAX_FIELD = float(FIELD["t_max_s"].min()) if FIELD["t_max_s"].notna().any() else None
        FIELD["limiting"] = FIELD["t_max_s"] == FIELD["t_max_s"].min()
        show = FIELD[["id", "role", "V", "raw_max", "over_limit", "t_max_s", "snr", "ok_snr", "limiting", "star_flag"]].rename(columns={"raw_max": "hellster_Pixel_ADU", "over_limit": "über_Grenze", "t_max_s": "t_max_s", "snr": "S/N (Probebild)", "ok_snr": "S/N≥100"})
        display(show.round(2))
        print(f"Begrenzender Stern: {FIELD.loc[FIELD['limiting'], 'id'].iloc[0]} ({FIELD.loc[FIELD['limiting'], 'role'].iloc[0]}) → längste Belichtung {T_MAX_FIELD:.2f} s")

## Schritt 3c — Scharf gegen defokussiert, Plate-Solve-Test

Hellere Ziele werden **bewusst unscharf** gestellt. Der Vergleich mit einem **scharfen Referenzbild** zeigt, was das bringt und kostet:
* **Spitzenwert** sinkt mit 1/FWHM² → entsprechend längere Belichtung bis zur Sättigungsgrenze (und weniger Szintillation); bei Bayer-Kameras verschwindet das Farbmuster unter dem Stern.
* **S/N** der schwachen Comps sinkt ungefähr mit 1/FWHM (himmelsbegrenzt) — Näherung für Gauß-Profile.
* **Plate-Solving:** löst ASTAP das defokussierte Bild noch? Mit dem scharfen Bild als Wahrheit wird es zusätzlich **künstlich verunschärft** (Gauß und Kreisring/„Donut“, `defocus_image`) und die Grenze gesucht (Tabelle). Grenzen der Simulation: KONZEPT §12.1.

Real: scharfes Bild in `TEST_DIR/scharf/` (oder `VARPHOT_TEST_DIR_SHARP`); die Bilder aus Schritt 3 sind der **geplante Fokus**. Demo: synthetisches scharfes Bild, Plate-Solve übersprungen (synthetische Sterne gibt es in keiner Sterndatenbank).

In [ ]:
FWHM_SHARP, DEFOCUS_FACTOR, TRADE, PLATE, SHARP_WCS = None, None, None, dict(tested=False), None
SHARP_DIR = Path(os.environ.get("VARPHOT_TEST_DIR_SHARP") or SESSION.path("probe", "scharf"))
SHARP = None                                                       # (Name, Belichtung, Bild)
if MODE == "demo":
    _t = frames[len(frames) // 2][1]
    SHARP = (f"synthetisch scharf {_t:g}s", _t, synth_frame(3.5 if BAYER else 2.5, _t, 99))
elif SHARP_DIR.is_dir() and sorted(SHARP_DIR.glob("*.fit*")):
    _p = sorted(SHARP_DIR.glob("*.fit*"))[0]
    _img, _hdr, SHARP_WCS = read_frame(_p)
    SHARP = (_p.name, exposure_from_header(_hdr) or 1.0, _img)
if SHARP is None:
    print(f"Kein scharfes Referenzbild ({SHARP_DIR}) — Vergleich und Plate-Solve-Test übersprungen. Ein scharf fokussiertes Bild mit gleicher Belichtung wie ein Probebild dorthin legen.")
else:
    def _metrics(img):
        work = green_plane(img, PATTERN) if BAYER else np.asarray(img, float)
        df = star_metrics(work, max_adu=MAX_ADU, fwhm_guess=3.0 if BAYER else 4.0)
        return df, fwhm_summary(df)
    _df_s, FS = _metrics(SHARP[2])
    _k = int(np.argmin([abs(t - SHARP[1]) for _, t, _ in frames]))                 # geplanter Fokus: Probebild mit ähnlichster Belichtung
    _df_d, FD = _metrics(frames[_k][2])
    FWHM_SHARP = FS["fwhm"]
    DEFOCUS_FACTOR = FD["fwhm"] / FS["fwhm"] if FS["fwhm"] and FD["fwhm"] else None
    _mult = 2 if BAYER else 1
    cmp_tab = pd.DataFrame({
        "scharf": dict(Bild=SHARP[0], FWHM_px=round(FS["fwhm"], 2), FWHM_hell=round(FS["fwhm_bright"], 2), FWHM_schwach=round(FS["fwhm_faint"], 2), n_Sterne=len(_df_s), hellster_ADU=round(_df_s["raw_max"].max()) if len(_df_s) else None, gesättigt=int(_df_s["saturated"].sum())),
        "geplanter Fokus": dict(Bild=frames[_k][0], FWHM_px=round(FD["fwhm"], 2), FWHM_hell=round(FD["fwhm_bright"], 2), FWHM_schwach=round(FD["fwhm_faint"], 2), n_Sterne=len(_df_d), hellster_ADU=round(_df_d["raw_max"].max()) if len(_df_d) else None, gesättigt=int(_df_d["saturated"].sum()))})
    display(cmp_tab)
    if DEFOCUS_FACTOR:
        TRADE = defocus_tradeoff(FS["fwhm"], FD["fwhm"])
        print(f"Defokus: FWHM {FS['fwhm'] * _mult:.1f} → {FD['fwhm'] * _mult:.1f} px im Rohbild (×{DEFOCUS_FACTOR:.2f}). Spitzenwert × {TRADE['peak_factor']:.2f} → gleiche Sättigungsgrenze bei × {TRADE['exposure_gain']:.1f} längerer Belichtung; "
              f"S/N der schwachen Sterne ≈ × {TRADE['snr_factor']:.2f} (Näherung, himmelsbegrenzt); Aperturfläche × {TRADE['aperture_area_factor']:.1f}")
        if DEFOCUS_FACTOR < 1.2:
            print("Hinweis: der geplante Fokus ist kaum unschärfer als das scharfe Bild — soll defokussiert werden?")
        if TRADE["snr_factor"] < 0.4:
            print(f"⚠ Starkes Defokussieren: schwache Comps verlieren etwa {100 * (1 - TRADE['snr_factor']):.0f} % S/N — Comps mit S/N ≥ 100 prüfen (Schritt 3b).")
    for tag, f in (("scharf", FS), ("geplant", FD)):
        if f["grows_with_brightness"]:
            print(f"Hinweis ({tag}): helle Sterne sind breiter als schwache (FWHM {f['fwhm_bright']:.1f} gegen {f['fwhm_faint']:.1f} px; Flügel der PSF) — Apertur nach dem Handbuch (größter Stern); Comps sind schwächer.")
    if MODE == "demo":
        _blur = float(np.sqrt(max(FD["fwhm"] ** 2 - FS["fwhm"] ** 2, 0.0)))
        _, SIM = _metrics(defocus_image(SHARP[2], _blur * _mult, "gauss"))
        print(f"DEMO: Das scharfe Bild, mit defocus_image um {_blur * _mult:.1f} px verunschärft, hat FWHM {SIM['fwhm']:.1f} px (geplant {FD['fwhm']:.1f}) — die Simulation trifft den Defokus.")
        print("Plate-Solve-Test: nur mit echten Bildern (synthetische Sterne gibt es in keiner Sterndatenbank). Grenzen an einem echten LCO-Bild: KONZEPT §12.1.")
    elif not ASTAP_OK:
        print("Plate-Solve-Test übersprungen: ASTAP nicht gefunden (Schritt 0).")
    else:
        from varphot.astrometry import defocus_solve_ladder, plate_solve_summary, solve_array
        _ar = lambda im: green_plane(im, PATTERN) if BAYER else np.asarray(im, float)
        _hint = dict(RA=RA_DEG, DEC=DEC_DEG)
        if SHARP_WCS is not None:                                              # WCS im Kopf des scharfen Bildes = Wahrheit und Skala
            _scale = float(np.sqrt(abs(np.linalg.det(SHARP_WCS.pixel_scale_matrix)))) * 3600.0 * (_mult if BAYER else 1)
            _fov = _ar(SHARP[2]).shape[0] * _scale / 3600.0 / (_mult if BAYER else 1)
        else:
            _fov = min(FIELD_DEG) if FIELD_DEG else None
        if _fov is None:
            print("Plate-Solve-Test übersprungen: Feldgröße unbekannt — in Schritt 1 die Brennweite eintragen (oder ein scharfes Bild mit WCS im Kopf verwenden).")
            _s_sharp = _s_plan = None
        else:
            _s_sharp = plate_solve_summary(solve_array(_ar(SHARP[2]), _hint, fov_deg=_fov, radius_deg=5.0), _ar(SHARP[2]).shape, None if BAYER else SHARP_WCS)
            _s_plan = plate_solve_summary(solve_array(_ar(frames[_k][2]), _hint, fov_deg=_fov, radius_deg=5.0), _ar(frames[_k][2]).shape, None if BAYER else SHARP_WCS)
        if _s_sharp is not None:
            PLATE = dict(tested=True, sharp_solved=_s_sharp["solved"], planned_solved=_s_plan["solved"], scale_arcsec=_s_plan.get("scale_arcsec") or _s_sharp.get("scale_arcsec"))
            print(f"Plate-Solve: scharfes Bild {'gelöst' if _s_sharp['solved'] else 'NICHT gelöst'}, geplanter Fokus {'gelöst' if _s_plan['solved'] else 'NICHT gelöst'}" + (f" (Skala {PLATE['scale_arcsec']} ″/px" + (f", Versatz der Bildmitte {_s_plan['offset_arcsec']}″ gegen den Kopf" if "offset_arcsec" in _s_plan else "") + ")" if PLATE["scale_arcsec"] else ""))
        if _s_plan is not None and not _s_plan["solved"]:
            print("⚠ Der geplante Fokus wird von ASTAP nicht gelöst: ein scharfes Bild derselben Position zum Lösen aufnehmen oder weniger defokussieren (KONZEPT §12.1).")
        if os.environ.get("VARPHOT_NB00_LADDER", "1") != "0" and _s_sharp is not None and _s_sharp["solved"]:
            _w_truth = SHARP_WCS if (SHARP_WCS is not None and not BAYER) else solve_array(_ar(SHARP[2]), _hint, fov_deg=_fov, radius_deg=5.0).wcs
            LADDER = defocus_solve_ladder(_ar(SHARP[2]), _w_truth, _hint, blurs=(0, 4, 8, 12, 20, 30), kinds=("gauss", "donut"), radius_deg=5.0)
            LADDER["FWHM_gesamt_px"] = np.hypot(FS["fwhm"], LADDER["blur_px"]).round(1)
            display(LADDER)
            _lim = {k: float(LADDER[(LADDER["kind"] == k) & LADDER["solved"]]["FWHM_gesamt_px"].max()) for k in ("gauss", "donut") if LADDER[(LADDER["kind"] == k) & LADDER["solved"]].shape[0]}
            PLATE["solved_up_to_fwhm_px"] = _lim
            PLATE["ladder"] = LADDER[["kind", "blur_px", "solved", "offset_arcsec"]].to_dict("records")
            print("Gelöst bis FWHM (Simulation): " + ", ".join(f"{k} {v:.0f} px" for k, v in _lim.items()) + f";  geplanter Fokus FWHM {FD['fwhm']:.1f} px (Ebene)")
            if _lim and FD["fwhm"] > min(_lim.values()):
                print("⚠ Der geplante Fokus liegt über der kleinsten gelösten Grenze (Donut ist schwerer als Gauß) — Plate-Solve auf einem schärferen Bild einplanen.")

## Schritt 4 — Empfehlung

* **Belichtung:** Aus jedem **ungesättigten** Probebild folgt die längste Zeit, bei der der hellste Stern unter der Obergrenze bleibt (lineare Skalierung); es gilt das Minimum. Wird das Ziel bei hellster Phase heller als im Probebild,
  wird entsprechend gekürzt (`TARGET_IS_BRIGHTEST`: das Ziel ist der hellste Stern im Feld). Empfohlen werden 80 % der längsten Zeit.
* **Aperturen:** 1,5 × FWHM der Ebene (Handbuch §6.1.1), Himmelsring nach §6.1.2.
* **Warnungen:** Bayer-FWHM im Rohbild < 8 px (zu scharf → defokussieren), alle Probebilder gesättigt, FWHM sehr groß.

In [ ]:
TARGET_IS_BRIGHTEST = True
WARN = []
FWHM_PX = float(np.median(FW_ALL)) if FW_ALL else float("nan")
FWHM_RAW = FWHM_PX * (2 if BAYER else 1)
usable = TAB[(TAB["gesättigt"] == False) & TAB["hellster_Stern_ADU"].notna()]
if usable.empty:
    T_MAX = None
    WARN.append("Alle Probebilder sind gesättigt (oder ohne Sterne): mit kürzerer Belichtung bzw. mehr Defokus neu aufnehmen.")
else:
    t_list = [recommend_exposure(r["hellster_Stern_ADU"], r["Belichtung_s"], LIMIT, r["Offset"]) for _, r in usable.iterrows()]
    T_MAX = float(min(t_list))
    if TARGET_IS_BRIGHTEST and V_MAX is not None and V_MEAN is not None and V_MAX < V_MEAN:
        T_MAX /= 10 ** (0.4 * (V_MEAN - V_MAX))
        print(f"Korrektur auf die hellste Phase (V {V_MEAN} → {V_MAX}): Belichtung × {10 ** (-0.4 * (V_MEAN - V_MAX)):.2f}")
if T_MAX_FIELD is not None:
    T_MAX = T_MAX_FIELD if T_MAX is None else min(T_MAX, T_MAX_FIELD)                # jeder Katalogstern zählt, nicht nur der hellste Stern des Bildes
T_REC = None if T_MAX is None else round(0.8 * T_MAX, 1 if T_MAX < 30 else 0)
if FIELD is not None and T_REC is not None:
    FIELD["ok_snr_rec"] = FIELD["snr"] * np.sqrt(T_REC / float(ref["Belichtung_s"])) >= 100
    V_COMP_MIN = comp_v_min(FIELD[FIELD["role"] == "comp"], float(ref["Belichtung_s"]), T_REC, LIMIT, offv)
    comps = FIELD[FIELD["role"] == "comp"]
    over = comps[comps["t_max_s"] < T_REC]
    faint = comps[~comps["ok_snr_rec"]]
    if len(over):
        WARN.append("Comps über der Obergrenze bei der empfohlenen Belichtung: " + ", ".join(over["id"]) + " (nicht verwenden oder kürzer belichten)")
    if len(faint):
        WARN.append("Comps mit S/N < 100 bei der empfohlenen Belichtung: " + ", ".join(faint["id"]) + " (Handbuch §6.3.1)")
    n_good = int((~comps["id"].isin(set(over["id"]) | set(faint["id"]))).sum())
    if n_good < 4:
        WARN.append(f"nur {n_good} brauchbare Comps im Probebild (mindestens 4 empfohlen)")
AP, AP_INFO = {}, {}
if np.isfinite(FWHM_PX):
    from varphot import optimal_aperture
    _cand = [v for v in DFS.values() if v[2]["n"] >= 3] or list(DFS.values())
    _gain = CAM.gain_e_per_adu if CAM and CAM.gain_e_per_adu else 1.0
    _rn = CAM.readout_noise_e if CAM and CAM.readout_noise_e else 0.0
    _target_sat = bool(any(len(v[0]) and bool(v[0]["saturated"].any()) for v in DFS.values()))                # gesättigtes Ziel: Flügel zählen mehr → 98 % statt 95 % des Flusses
    _all = [optimal_aperture(w_, d_, f_["fwhm"], gain=_gain, read_noise_e=_rn, fraction=0.98 if _target_sat else 0.95) for d_, w_, f_ in _cand if np.isfinite(f_["fwhm"])]
    _all.sort(key=lambda a_: a_["ap_r"])
    AP_INFO = _all[(len(_all) - 1) // 2]                                                                              # MEDIAN über die Probebilder: ein Ausreißer (sehr schlechtes Seeing) bestimmt nicht die Apertur der ganzen Reihe
    if len(_all) > 1:
        print(f"Apertur je Probebild: {[a_['ap_r'] for a_ in _all]} px → Median {AP_INFO['ap_r']} px")
    AP = {k: AP_INFO[k] for k in ("ap_r", "ap_r_in", "ap_r_out")}
    if AP_INFO["method"] == "growth+snr":
        print(f"Apertur aus den Daten (Wachstumskurve 95 % = {AP_INFO['r_growth']:.1f} px, S/N-Optimum der schwachen Sterne = {AP_INFO['r_snr']:.1f} px): {AP['ap_r']} px "
              f"(S/N {100 * AP_INFO['snr_ratio']:.0f} % des Optimums; die Handbuch-Regel 1,5 × FWHM = {AP_INFO['handbook_r']} px gäbe {100 * AP_INFO['snr_ratio_handbook']:.0f} %)")
    for _n in AP_INFO.get("notes", []):
        print("Hinweis:", _n)
    AP_INFO = {k: (round(v, 3) if isinstance(v, float) else v) for k, v in AP_INFO.items() if k not in ("radii", "growth", "snr")}
if BAYER and np.isfinite(FWHM_RAW) and FWHM_RAW < MIN_RAW_FWHM_BAYER:
    WARN.append(f"FWHM im Rohbild {FWHM_RAW:.1f} px < {MIN_RAW_FWHM_BAYER:.0f} px: Sterne zu scharf für die Bayer-Kanäle (Kanalschwankungen je nach Pixelfarbe unter dem Sternzentrum) — defokussieren (Handbuch §5.5.1).")
if np.isfinite(FWHM_PX) and FWHM_PX > 25:
    WARN.append(f"FWHM {FWHM_PX:.0f} px sehr groß: Sterne evtl. ringförmig (Gauß-Anpassung unsicher), Himmelsring und Nachbarsterne prüfen.")
if len(FW_ALL) >= 2 and np.std(FW_ALL) / np.mean(FW_ALL) > 0.15:
    WARN.append("FWHM schwankt zwischen den Probebildern um > 15 % — Fokus/Seeing/Nachführung instabil?")
print(f"FWHM {FWHM_PX:.2f} px in der ausgewerteten Ebene" + (f" = {FWHM_RAW:.1f} px im Rohbild" if BAYER else ""))
print(f"Aperturen (Pixel der {'Grün-' if BAYER else ''}Ebene): {AP}")
if V_COMP_MIN is not None:
    print(f"Hellster zulässiger Comp bei {T_REC} s: V ≥ {V_COMP_MIN:.2f} (hellere Comps sättigen/überschreiten die Obergrenze) — NB03 übernimmt das über den Plan")
if FWHM_SHARP and T_MAX is not None and TRADE is not None:
    print(f"Ohne Defokus (scharf, FWHM {FWHM_SHARP * (2 if BAYER else 1):.1f} px) wäre die längste Belichtung etwa {T_MAX / TRADE['exposure_gain']:.1f} s (Gauß-Näherung) — Defokus bringt × {TRADE['exposure_gain']:.1f}.")
T_WANT = float(os.environ["VARPHOT_T_WANT"]) if os.environ.get("VARPHOT_T_WANT") else None
if T_WANT and not usable.empty and np.isfinite(FWHM_PX):
    _r = usable.sort_values("hellster_Stern_ADU", ascending=False).iloc[0]
    _need = defocus_needed(_r["hellster_Stern_ADU"], _r["Offset"], LIMIT, FWHM_PX, _r["Belichtung_s"], T_WANT)
    print(f"Für {T_WANT:g} s Belichtung braucht der hellste Stern FWHM ≥ {_need * (2 if BAYER else 1):.1f} px im Rohbild (jetzt {FWHM_RAW:.1f} px)" + (" — ausreichend." if _need <= FWHM_PX * 1.02 else " — mehr defokussieren."))
print(f"Längste Belichtung bis zur Obergrenze ({LIMIT:.0f} ADU): {'—' if T_MAX is None else f'{T_MAX:.1f} s'};  empfohlen: {'—' if T_REC is None else f'{T_REC} s'}")
for w in WARN:
    print("⚠", w)

In [ ]:
from IPython.display import display, HTML as _HTML

def _tbl(rows, header=None):
    h = f"<tr><th>{'</th><th>'.join(header)}</th></tr>" if header else ""
    body = "".join(f"<tr><td>{'</td><td>'.join(str(c) for c in r)}</td></tr>" for r in rows)
    return f"<table style='border-collapse:collapse;font-size:.9em'>{h}{body}</table>"

_ebene = "Grün-Ebene" if BAYER else "Ebene"
_rows_opt = [
    ("FWHM", f"{FWHM_PX:.2f} px ({_ebene})" + (f" = {FWHM_RAW:.1f} px Rohbild" if BAYER else "")),
    ("Aperturradius ap_r", f"{AP.get('ap_r','—')} px"),
    ("Himmelsring", f"ap_r_in {AP.get('ap_r_in','—')} px · ap_r_out {AP.get('ap_r_out','—')} px"),
]
if AP_INFO.get("method") == "growth+snr":
    _rows_opt.append(("Apertur-Methode",
        f"Wachstumskurve {AP_INFO.get('r_growth','?'):.1f} px · S/N-Opt {AP_INFO.get('r_snr','?'):.1f} px → "
        f"{AP.get('ap_r','?')} px (S/N {100*AP_INFO.get('snr_ratio',0):.0f}% des Optimums)"))

_rows_bel = [
    ("Obergrenze", f"{LIMIT:.0f} ADU"),
    ("Längste Belichtung", f"{'—' if T_MAX is None else f'{T_MAX:.1f} s'}"),
    ("Empfohlene Belichtung", f"{'—' if T_REC is None else f'{T_REC} s'}"),
]
if TRADE is not None and T_MAX is not None:
    _rows_bel.append(("Ohne Defokus wäre T_max", f"≈ {T_MAX/TRADE['exposure_gain']:.1f} s (Defokus × {TRADE['exposure_gain']:.1f})"))
if V_COMP_MIN is not None:
    _rows_bel.append(("Hellster Comp (sicher)", f"V ≥ {V_COMP_MIN:.2f} bei {T_REC} s"))

_warn_html = "".join(
    f"<tr><td style='color:#b45309;padding:2px 6px'>⚠ {w}</td></tr>" for w in WARN
) or "<tr><td style='color:green'>Keine Warnungen</td></tr>"

html = f"""
<h4 style='margin:8px 0 4px'>Optik &amp; Aperturen</h4>
{_tbl(_rows_opt, ["Größe","Wert"])}
<h4 style='margin:12px 0 4px'>Belichtung</h4>
{_tbl(_rows_bel, ["Größe","Wert"])}
<h4 style='margin:12px 0 4px'>Warnungen</h4>
<table style='border-collapse:collapse;font-size:.9em'>{_warn_html}</table>
"""
display(_HTML(html))


## Schritt 5 — Plan speichern

Im Modus `real` wird der Plan in `PHOTOMETRIE_PY_DATA/plans/` abgelegt. NB04 liest ihn über `VARPHOT_PLAN=<Datei>` und übernimmt die Aperturen; die Methode (Meldefilter, `OBSTYPE`) steht für NB06 bereit. Im Demo-Modus wird nichts gespeichert.

In [ ]:
PLAN = ObservingPlan(target=TARGET, ra_deg=RA_DEG, dec_deg=DEC_DEG, v_mean=V_MEAN, v_max=V_MAX, method=METHOD, aavso_filter=AAVSO_FILTER, obstype=OBSTYPE,
                     camera_model=MODEL, camera_setting=SETTING, bayer_pattern=PATTERN if BAYER else "", site_name=SITE.name if SITE else "",
                     fwhm_px=None if not np.isfinite(FWHM_PX) else round(FWHM_PX, 2), fwhm_raw_px=None if not np.isfinite(FWHM_RAW) else round(FWHM_RAW, 2),
                     aperture=AP, offset_adu=OFFSET, max_adu=MAX_ADU, limit_adu=LIMIT, limit_source=LIMIT_SRC,
                     exptime_max_s=None if T_MAX is None else round(T_MAX, 2), exptime_recommended_s=T_REC, warnings=WARN,
                     v_comp_min=None if V_COMP_MIN is None else round(V_COMP_MIN, 2), moon=MOON, focal_length_mm=FOCAL_MM, field_deg=FIELD_DEG, fwhm_sharp_px=None if FWHM_SHARP is None else round(float(FWHM_SHARP), 2), defocus_factor=None if DEFOCUS_FACTOR is None else round(float(DEFOCUS_FACTOR), 2),
                     plate_solve=PLATE, readiness=READINESS, aperture_info=AP_INFO,
                     field_checks=[] if FIELD is None else FIELD[["id", "role", "V", "raw_max", "t_max_s", "snr", "ok_snr", "limiting"]].round(3).to_dict("records"),
                     notes=f"{len(frames)} Probebilder, {'Demo' if MODE == 'demo' else TEST_DIR}")
if MODE == "real":
    path = save_plan(PLAN)
    _sharp = sorted(SHARP_DIR.glob("*.fit*")) if SHARP_DIR.is_dir() else []
    SESSION.update(plan=str(path), tag=session_tag(TARGET, METHOD, datetime.date.today().isoformat()), ref_frame=str(_sharp[0]) if _sharp else SESSION.ref_frame,
                   ref_fwhm_px=None if not np.isfinite(FWHM_PX) else float(FWHM_PX), frames_dir=SESSION.frames_dir or str(SESSION.path("process")))
    SESSION.update(results=str(SESSION.path("results", SESSION.tag)))
    print("Plan gespeichert:", path, "\nSitzung gespeichert:", SESSION.file, "\n→ weiter mit 01_vorbereitung.ipynb (liest die Sitzung automatisch)")
else:
    print("DEMO: nichts gespeichert. Plan wäre:"); display(pd.Series({k: v for k, v in PLAN.__dict__.items()}))